In [1]:
from datasets import load_dataset

# 1. Load the dataset directly from the Hugging Face Hub
#    You may need to log in first: huggingface-cli login
print("Loading dataset...")
dataset = load_dataset("strakammm/generals_io_replays")

# The dataset is loaded into a DatasetDict, we'll use the 'train' split
train_dataset = dataset['train']

# 2. Print the total number of replays in the dataset
print(f"\nTotal number of replays: {len(train_dataset)}")

# 3. Iterate over the first few replays to showcase the data
print("\n--- Showcasing first 5 replays ---")
for i in range(5):
    replay = train_dataset[i]
    
    # Get the players and the number of moves
    players = replay['usernames']
    num_moves = len(replay['moves'])
    
    print(f"\nReplay {i+1}:")
    print(f"  Players: {players[0]} vs {players[1]}")
    print(f"  Total moves: {num_moves}")

print("\n------------------------------------")

/Users/aryamanwade/Desktop/ml_projects/generals_ai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dataset...


Generating train split: 100%|██████████| 18803/18803 [00:00<00:00, 67721.75 examples/s]


Total number of replays: 18803

--- Showcasing first 5 replays ---

Replay 1:
  Players: +qun771923864 vs tototatatonton
  Total moves: 428

Replay 2:
  Players: Human.exe vs lishuo
  Total moves: 736

Replay 3:
  Players: Russian Invasion vs Human.exe
  Total moves: 1089

Replay 4:
  Players: wjy love byb vs hshshssss
  Total moves: 372

Replay 5:
  Players: csallen vs genshinrals
  Total moves: 298

------------------------------------


DATA SET SCHEMA 

The structure of each JSON object is as follows:

Field	Type	Description
version	Integer	The version number of the replay format.
id	String	A unique identifier for the replay.
mapWidth	Integer	The width of the game map in tiles.
mapHeight	Integer	The height of the game map in tiles.
usernames	List[String]	A list of the usernames for the players in the game. The index corresponds to the player's turn order.
stars	List[Integer]	The star rating of each player at the time the game was played.
cities	List[Integer]	A list of tile indices representing the locations of all neutral cities at the start of the game.
cityArmies	List[Integer]	The initial army count for each corresponding city in the cities list.
generals	List[Integer]	The starting tile indices for each player's general (home base).
mountains	List[Integer]	A list of tile indices for all impassable mountain tiles on the map.
moves	List[List[Integer]]	The core sequence of game actions. Each inner list is a move tuple, typically representing [player_index, start_tile, end_tile, is_50%_move, turn_number].
afks	List[]	A list that would contain information on AFK players. This has been filtered and will always be empty.

In [2]:
import random

idx = random.randint(0, len(train_dataset) - 1)
sample = train_dataset[idx]

print(f"Random sample index: {idx}")
print(f"Columns: {list(sample.keys())}\n")
print("--- All column values ---\n")

for col, value in sample.items():
    print(f"{col}:")
    print(f"  {value}")
    print()


Random sample index: 18060
Columns: ['version', 'id', 'mapWidth', 'mapHeight', 'usernames', 'stars', 'cities', 'cityArmies', 'generals', 'mountains', 'moves']

--- All column values ---

version:
  13

id:
  uD4aX3WAF

mapWidth:
  19

mapHeight:
  18

usernames:
  ['zzd_bot release', 'Human.exe']

stars:
  [76, 74]

cities:
  [255, 79, 221, 99, 310, 307, 339, 22, 315, 77]

cityArmies:
  [49, 41, 40, 46, 49, 49, 49, 47, 42, 43]

generals:
  [20, 257]

mountains:
  [5, 10, 12, 14, 15, 18, 23, 26, 27, 32, 33, 37, 41, 49, 56, 58, 64, 66, 70, 75, 76, 81, 86, 89, 90, 91, 92, 110, 111, 117, 123, 136, 146, 152, 156, 174, 177, 186, 197, 199, 200, 203, 208, 210, 215, 226, 228, 235, 236, 238, 258, 262, 265, 279, 281, 284, 287, 293, 298, 303, 304, 320, 322, 324, 326, 327, 331, 333]

moves:
  [[1, 257, 276, 0, 22], [1, 276, 275, 0, 23], [0, 20, 39, 0, 24], [1, 275, 274, 0, 24], [1, 274, 273, 0, 25], [0, 39, 40, 0, 25], [0, 40, 59, 0, 26], [1, 273, 272, 0, 26], [1, 272, 271, 0, 27], [0, 59, 60, 0, 2

OBS AND ACTION FOR THE ENV

Observation
Per player: Observation NamedTuple (fog unless perfect_info=True).

Field	Shape	Type
armies
(H, W)
int
generals, cities, mountains
(H, W)
bool mask
owned_cells, opponent_cells, neutral_cells
(H, W)
bool
fog_cells, structures_in_fog
(H, W)
bool
owned_land_count, owned_army_count
()
scalar
opponent_land_count, opponent_army_count
()
scalar
timestep
()
scalar
NN form: obs.as_tensor() → (14, H, W)
channels: armies, generals, cities, mountains, neutral, owned, opponent, fog, structures_in_fog, owned_land, owned_army, opp_land, opp_army, timestep

GeneralsEnv.step stacks both players: obs fields shaped (2, H, W) / scalars (2,).

Action
int32 length-5: [pass, row, col, direction, split]

Idx	Name	Values
0
pass
0 move, 1 pass
1
row
source row
2
col
source col
3
direction
0↑ 1↓ 2← 3→
4
split
0 = all-but-1, 1 = half
Env step input: (2, 5) — one action per player.
Batched: (N, 2, 5).

Legal moves: compute_valid_move_mask(...) → (H, W, 4).